# Machine Learning Analysis of Factors Influencing Merger Cancellations

End-to-end walkthrough: data → cleaning → features → models → evaluation → explainability → prediction.

> Probability-based **decision-support** tool. It does **not** guarantee any individual deal's outcome.

Run the cells top-to-bottom. The heavy lifting lives in the `src/` package so this notebook stays readable.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config
pd.set_option('display.max_columns', 60)

## 1. Generate (or load) data and run the full pipeline
If you have real data, drop it at `data/raw/ma_deals_raw.csv` (see the template) and skip generation.

In [ ]:
# One-liner: runs every stage and writes all artifacts to reports/ and models/.
# !python -m src.run_pipeline --n 5000
from src import generate_data, data_cleaning, feature_engineering
raw = generate_data.generate(5000); raw.to_csv(config.RAW_CSV, index=False)
clean = data_cleaning.clean(raw)
feats = feature_engineering.build_features(clean)
feats.head()

## 2. Target balance & cancellation rate by election context

In [ ]:
print('Completion rate:', feats[config.TARGET_COL].mean().round(3))
feats['cancelled'] = 1 - feats[config.TARGET_COL]
for label, mask in {
    'Election year': feats.election_year==1,
    'Transition window': feats.transition_year==1,
    'New-president year': feats.new_president_year==1,
    'Ordinary year': (feats.election_year==0)&(feats.transition_year==0)&(feats.new_president_year==0),
}.items():
    print(f'{label:20s} cancel rate = {feats.loc[mask, "cancelled"].mean():.1%}  (n={mask.sum()})')

## 3. Train & compare models

In [ ]:
feats.to_csv(config.FEATURES_CSV, index=False)
from src import train
results, best = train.train_and_compare()
results

## 4. Evaluate & explain

In [ ]:
from src import evaluate, explain
evaluate.evaluate(); explain.main()
from IPython.display import Image, display
for fig in ['confusion_matrix','roc_curve','feature_importance','election_cancellation_rates','election_vs_business']:
    display(Image(str(config.FIGURES_DIR / f'{fig}.png')))

## 5. Predict a proposed future merger

In [ ]:
from src.predict import predict_deal
predict_deal({
    'announcement_date': '2028-03-15',
    'deal_value_musd': 42000,
    'industry': 'Technology', 'sector': 'Semiconductors',
    'target_country': 'US', 'acquirer_country': 'China',
    'payment_method': 'stock', 'regulatory_review': 'extended',
    'rumor_or_leak': 'yes', 'expected_days_to_close': 300,
})